# Landslide Runout Emulator — Uncertainty Demo

**Goal.** Show a fast workflow to predict landslide runout and propagate input uncertainty.


In [ ]:
from runout_utils import prepare_case_chip
import matplotlib

matplotlib.rcParams["mathtext.fontset"] = "stix"
matplotlib.rcParams["font.family"] = "STIXGeneral"
matplotlib.rcParams["figure.figsize"] = (6, 4)

case = 'Maoxian'

Crop the Digital Elevation Model centered to the .kml point. Make sure the dem name is == to case.

In [ ]:
chip, transform, profile, paths = prepare_case_chip(
    case=case,
    root=".",          
    size=256,
    save=True,
    plot=True,
)

# Generate predictions
Check if GPU is available

In [ ]:
import torch
import numpy as np
from emulator import *

if torch.cuda.is_available():
    print("GPU is available.")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available.")

Run a single baseline simulation with custom parameters and visualize predicted runout mask and thickness over the DEM hillshade.

In [ ]:
baseline = dict(volume=9e6, cohesion=25000.0, rho=1800.0)

run_landslide_batch(
    landslide=case,
    image_size=(256, 256),
    cell_size=30.0,
    cohesions=[baseline["cohesion"]],
    rhos=[baseline["rho"]],
    volumes=[baseline["volume"]],
    model_path="model/h/best.pth",
    device="cuda",                 # "cpu" or "cuda" or "mps"
    plot=True,
    combination_mode="elementwise",   
    reuse_dist=True,                  
)

Generate Sobol samples across the parameter ranges (volume, cohesion, density) to explore uncertainty in a structured way.

In [ ]:
N = 1024  

samples = sobol_params(
    n=N,
    volume_range=(5e6, 1e7),
    cohesion_range=(5e3, 5e4),
    rho_range=(1600, 2400),
    seed=123,
    log_volume=True,
    
)
samples.head()

Run the emulator once for each Sobol triplet (elementwise mode) to generate a full ensemble of runout predictions across the sampled parameter space.

In [ ]:
# Batch run
C = samples["cohesion"].to_numpy(float)
R = samples["rho"].to_numpy(float)
V = samples["volume"].to_numpy(float)

run_landslide_batch(
    landslide=case,
    image_size=(256, 256),
    cell_size=30.0,
    cohesions=C, rhos=R, volumes=V,
    model_path="model/h/best.pth",
    device="cuda",                 # "cpu" or "cuda" or "mps"
    plot=False,
    combination_mode="elementwise",   # elementwise for Sobol triplets
    reuse_dist=True,                  # fast path if source footprint is fixed
)

Aggregate the ensemble of runouts into an exceedance probability map, save it as GeoTIFF, and plot it over the DEM hillshade.

In [ ]:
from runout_utils import *

res = save_probability_and_plot(
    case=case,
    threshold=0.05,
    match_shape="strict", 
    alpha=0.6,
    cmap="viridis",
)

Post-process the ensemble: compute exceedance probability, save as GeoTIFF and PNG overlay, and record metadata for reproducibility.

In [ ]:
import os
from runout_utils import *

out_dir = f"{case}/output_nn"
dem = f"{case}/dem.tif"
thr = 0.1

stack = load_runout_stack(out_dir, match_shape="strict")
prob = exceedance_probability(stack, threshold=thr)

prob_tif = os.path.join(out_dir, "prob_thr010.tif")
save_geotiff(prob, dem, prob_tif)
overlay_on_hillshade(prob, dem, title=f"{case} – P(runout>{thr})", out_png=os.path.join(out_dir, "prob_thr010.png"))

save_metadata_json(os.path.join(out_dir, "prob_thr010.meta.json"), {
    "case": case, "threshold": thr, "n_runs": int(stack.shape[0]), "shape": list(prob.shape)
})

Derive quantile maps (Q50, Q90) from the ensemble thickness predictions and overlay them on the DEM hillshade to visualize median vs. high-end scenarios.

In [ ]:
from runout_utils import percentile_maps

p50, p90 = percentile_maps(stack, [0.5, 0.9])
save_geotiff(p50, dem, os.path.join(out_dir, "q50.tif"))
save_geotiff(p90, dem, os.path.join(out_dir, "q90.tif"))
overlay_on_hillshade(p50, dem, title=f"{case} – Q50", out_png=os.path.join(out_dir, "q50.png"))
overlay_on_hillshade(p90, dem, title=f"{case} – Q90", out_png=os.path.join(out_dir, "q90.png"))